In [1]:
!git clone https://github.com/sumit760/Codeglue.git

Cloning into 'Codeglue'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 23 (delta 4), reused 2 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 29.50 KiB | 5.90 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
%cd /kaggle/working
!ls

/kaggle/working
Codeglue  __notebook__.ipynb


In [3]:
!git config user.name "PrudhviGowroju"
!git config user.email "Prudhvi90hrithik@gmail.com"
!git config --list | grep user

fatal: not in a git directory
fatal: not in a git directory


In [4]:
!git status


fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [5]:
!git checkout Prudhvi

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [6]:
!git branch

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [7]:
# 1. Install/upgrade necessary libraries
!pip install -q transformers datasets wandb evaluate scikit-learn

from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login
from datasets import load_dataset

# 2. Authenticate using Kaggle Secrets
try:
    secrets = UserSecretsClient()
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    login(token=secrets.get_secret('HF_TOKEN'))
    wandb.login()
    print("✅ Successfully logged into Hugging Face and Weights & Biases!")
except Exception as e:
    print(f"⚠️ Secret loading error: {e}")

# 3. Load the dataset
print("\nDownloading the CodeXGLUE dataset...")
dataset = load_dataset("google/code_x_glue_cc_defect_detection")

# 4. Inspect the data structure
print("\n--- DATASET STRUCTURE ---")
print(dataset)

print("\n--- SAMPLE ITEM (Row 0) ---")
print(dataset['train'][0])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


⚠️ Secret loading error: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 122006935 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.



README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21854 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2732 [00:00<?, ? examples/s]


--- DATASET STRUCTURE ---
DatasetDict({
    train: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 21854
    })
    validation: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 2732
    })
    test: Dataset({
        features: ['id', 'func', 'target', 'project', 'commit_id'],
        num_rows: 2732
    })
})

--- SAMPLE ITEM (Row 0) ---
{'id': 0, 'func': 'static av_cold int vdadec_init(AVCodecContext *avctx)\n\n{\n\n    VDADecoderContext *ctx = avctx->priv_data;\n\n    struct vda_context *vda_ctx = &ctx->vda_ctx;\n\n    OSStatus status;\n\n    int ret;\n\n\n\n    ctx->h264_initialized = 0;\n\n\n\n    /* init pix_fmts of codec */\n\n    if (!ff_h264_vda_decoder.pix_fmts) {\n\n        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)\n\n            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;\n\n        else\n\n            ff_h264_vda_decoder.pix_fmts = vda_pix

In [8]:
import json
from transformers import AutoTokenizer

# 1. Load the tokenizer for CodeBERTa
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define the preprocessing function
def preprocess_function(examples):
    # Clean the code: replace multiple spaces/newlines with a single space
    cleaned_code = [" ".join(code.split()) for code in examples["func"]]
    
    # Tokenize the code (truncating to 512 tokens to fit BERT's maximum length)
    model_inputs = tokenizer(
        cleaned_code,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    
    # Convert Boolean target to integers (0 = Safe, 1 = Defective)
    model_inputs["labels"] = [1 if label else 0 for label in examples["target"]]
    
    return model_inputs

# 3. Apply the processing to all splits and drop old columns
print("Cleaning, tokenizing, and dropping unused columns. This may take a minute...")
columns_to_remove = ['id', 'func', 'target', 'project', 'commit_id']
tokenized_dataset = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=columns_to_remove
)

# 4. Create and save the id2label mapping file
id2label = {0: "Safe", 1: "Defective"}

with open("id2label.json", "w") as f:
    json.dump(id2label, f)

print("✅ Data preparation complete! id2label.json is saved locally.")
print("\n--- TOKENIZED DATASET STRUCTURE ---")
print(tokenized_dataset)

Loading tokenizer for huggingface/CodeBERTa-small-v1...


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Cleaning, tokenizing, and dropping unused columns. This may take a minute...


Map:   0%|          | 0/21854 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

Map:   0%|          | 0/2732 [00:00<?, ? examples/s]

✅ Data preparation complete! id2label.json is saved locally.

--- TOKENIZED DATASET STRUCTURE ---
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21854
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2732
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2732
    })
})


In [9]:
import wandb
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Load the Model
model_name = "huggingface/CodeBERTa-small-v1"
print(f"Loading {model_name} for sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Safe", 1: "Defective"},
    label2id={"Safe": 0, "Defective": 1}
)

# 2. Define Metrics (Accuracy and F1)
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

# 3. Initialize W&B for Version 1
wandb.init(
    project="mlops-assignment3", 
    name="run-v1", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "version": "v1",
        "platform": "Kaggle"
    }
)

# 4. Set Training Arguments
training_args = TrainingArguments(
    output_dir='./results_v1',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v1",
    fp16=True # Accelerates training on Kaggle GPUs
)

# 5. Initialize Trainer and Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v1...")
trainer.train()

# 6. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset['test'])
print(test_results)

wandb.finish()

Loading huggingface/CodeBERTa-small-v1 for sequence classification...


pytorch_model.bin:   0%|          | 0.00/336M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/336M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: huggingface/CodeBERTa-small-v1
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: [wa

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Evaluating on test set...


wandb: updating run metadata


{'eval_loss': 1.2710812091827393, 'eval_accuracy': 0.6259150805270863, 'eval_f1': 0.6264784846971861, 'eval_runtime': 24.9098, 'eval_samples_per_second': 109.676, 'eval_steps_per_second': 3.452, 'epoch': 3.0}


wandb: uploading history steps 8-8, summary, console lines 13-13
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▇█▆
wandb:                 eval/f1 ▁▇█▆
wandb:               eval/loss ▂▁█▂
wandb:            eval/runtime ▁▅▄█
wandb: eval/samples_per_second █▄▅▁
wandb:   eval/steps_per_second █▄▅▁
wandb:             train/epoch ▁▂▃▅▆████
wandb:       train/global_step ▁▂▃▅▆████
wandb:         train/grad_norm ▁▃█▃
wandb:     train/learning_rate █▆▃▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.62592
wandb:                 eval/f1 0.62648
wandb:               eval/loss 1.27108
wandb:            eval/runtime 24.9098
wandb: eval/samples_per_second 109.676
wandb:   eval/steps_per_second 3.452
wandb:              total_flos 8684827590684672.0
wandb:             train/epoch 3
wandb:       train/global_step 2049
wandb:         train/grad_norm 13.51553
wandb:                      +6 ...
wandb: 
wandb: 🚀 View run run-v1 at: https://wan

In [10]:
# 1. Initialize W&B for Version 2
wandb.init(
    project="mlops-assignment3", 
    name="run-v2", 
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 5e-5, # HYPERPARAMETER CHANGE
        "version": "v2",
        "platform": "Kaggle"
    }
)

# 2. Set Training Arguments for v2
training_args_v2 = TrainingArguments(
    output_dir='./results_v2',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5, # HYPERPARAMETER CHANGE
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb", 
    run_name="run-v2",
    fp16=True 
)

# 3. Initialize Trainer and Train
trainer_v2 = Trainer(
    model=model,
    args=training_args_v2,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics
)

print("Starting training run v2...")
trainer_v2.train()

# 4. Evaluate on Test Set and Finish W&B
print("\nEvaluating on test set for run v2...")
test_results_v2 = trainer_v2.evaluate(tokenized_dataset['test'])
print(test_results_v2)

wandb.finish()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260609_103418-k0q63qjv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run run-v2
wandb: ⭐️ View project at https://wandb.ai/prudhvigowroju_iitj/mlops-assignment3
wandb: 🚀 View run at https://wandb.ai/prudhvigowroju_iitj/mlops-assignment3/runs/k0q63qjv


Starting training run v2...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.071466,1.541576,0.613470,0.614157
2,0.868216,1.568157,0.631772,0.627530
3,0.636727,1.895001,0.629941,0.630298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Evaluating on test set for run v2...


wandb: updating run metadata


{'eval_loss': 1.5756773948669434, 'eval_accuracy': 0.6050512445095169, 'eval_f1': 0.604173556256138, 'eval_runtime': 24.9324, 'eval_samples_per_second': 109.576, 'eval_steps_per_second': 3.449, 'epoch': 3.0}


wandb: uploading history steps 8-8, summary, console lines 13-13
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▃██▁
wandb:                 eval/f1 ▄▇█▁
wandb:               eval/loss ▁▂█▂
wandb:            eval/runtime █▇▁█
wandb: eval/samples_per_second ▁▂█▁
wandb:   eval/steps_per_second ▁▃█▁
wandb:             train/epoch ▁▂▃▅▆████
wandb:       train/global_step ▁▂▃▅▆████
wandb:         train/grad_norm ▁▂█▁
wandb:     train/learning_rate █▆▃▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.60505
wandb:                 eval/f1 0.60417
wandb:               eval/loss 1.57568
wandb:            eval/runtime 24.9324
wandb: eval/samples_per_second 109.576
wandb:   eval/steps_per_second 3.449
wandb:              total_flos 8684827590684672.0
wandb:             train/epoch 3
wandb:       train/global_step 2049
wandb:         train/grad_norm 11.63667
wandb:                      +6 ...
wandb: 
wandb: 🚀 View run run-v2 at: https://wan